In [2]:
import os
import sys
import random
import wandb
import argparse
import numpy as np
import tensorflow as tf
import keras
import pprint

from wandb.integration.keras import WandbMetricsLogger

from src.constant import PROJECT_NAME
from src.utils import filter_vocabulary, load_yaml, store_yaml, load_config
from src.preprocessing import construct_features_meta
from src.preprocessing.data_loader import load_data
from src.sampler import BayesianSampler
from src.losses.bayesian_personalized_ranking import BayesianPersonalizedRankingLoss
from src.models.matrix_factorization import MatrixFactorization

In [13]:
# Set up configuration
config = {
    "random_seed": 42,
    "model": "matrix_factorization",
    "embedding_dimension": 16,
    "l1_regularization": 0.0,
    "l2_regularization": 0.0,
    "embedding_dropout_rate": 0.0,
    "max_epoch": 2,
    "learning_rate": 0.01,
    "batch_size": 16384,
    "shuffle": False,
    "early_stopping": False,
    "evaluation_cutoffs": [2, 10, 50],
}

# Set random seeds for reproducibility
random.seed(config["random_seed"])
np.random.seed(config["random_seed"])
tf.random.set_seed(config["random_seed"])    

In [6]:
# A. Load and preprocess data
train_user_interaction = load_data("dataset/yelp2018/train.txt")
train_features_meta = construct_features_meta(train_user_interaction)
test_user_interaction = load_data("dataset/yelp2018/test.txt")
test_features_meta = construct_features_meta(test_user_interaction)

user_items = train_user_interaction.groupby("user_id")["item_id"].apply(set).to_dict()
item_users = train_user_interaction.groupby("item_id")["user_id"].apply(set).to_dict()

train_dataset = tf.data.Dataset.from_tensor_slices(
{
        "user_id": train_user_interaction["user_id"].values,
        "item_id": train_user_interaction["item_id"].values
    }
)
test_dataset = tf.data.Dataset.from_tensor_slices(
    {
        "user_id": test_user_interaction["user_id"].values,
        "item_id": test_user_interaction["item_id"].values
    }
)
print(f"{'='*10} Dataset Summary {'='*21}")
print(f"Training Dataset: {len(train_dataset)}")
print(f"Test Dataset: {len(test_dataset)}")
pprint.pprint(filter_vocabulary(train_features_meta))
print(f"{'='*48}")

========== Dataset Summary =====================
Training Dataset: 1237259
Test Dataset: 324147
{'item_id': {'dtype': 'int64', 'unique_count': 38048},
 'user_id': {'dtype': 'int64', 'unique_count': 31668}}


2026-02-28 14:11:16.041851: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1772287876.043131 1812672 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20797 MB memory:  -> device: 0, name: NVIDIA A10G, pci bus id: 0000:00:1e.0, compute capability: 8.6


In [14]:
# B. Model Initialization
sampler = BayesianSampler(item_set=train_features_meta["item_id"]["vocabulary"], user_items=user_items)

if config["model"] == "matrix_factorization":
    model = MatrixFactorization(
        train_features_meta, 
        embedding_dimension_count=config["embedding_dimension"],
        l1_regularization=config["l1_regularization"],
        l2_regularization=config["l2_regularization"],
        embedding_dropout_rate=config["embedding_dropout_rate"],
        evaluation_cutoffs=config["evaluation_cutoffs"]
    )
else:
    raise ValueError(f"Unknown model type: {config['model']}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
    loss_functions=[
        BayesianPersonalizedRankingLoss()
    ],
    sampler=sampler,
)

In [15]:
# C. Model Training
callbacks = []
if config["early_stopping"]:
    callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor=config["early_stopping_monitor"],
                mode=config["early_stopping_mode"],
                patience=config["early_stopping_patience"],
                restore_best_weights=True,
                verbose=1
        )
    )

results = model.fit(
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    nepochs=config["max_epoch"],
    shuffle=config["shuffle"],
    batch_size=config["batch_size"],
    callbacks=callbacks
)

print(f"{'='*10} Final Results {'='*24}")
pprint.pprint(results)
print(f"{'='*48}")

OE [1/2]:   0%|                                                                                                                                       | 0/31668 [00:00<?, ?it/s]2026-02-28 14:15:47.412532: I external/local_xla/xla/service/service.cc:163] XLA service 0x40dfd100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-28 14:15:47.412559: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A10G, Compute Capability 8.6
2026-02-28 14:16:06.788202: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
I0000 00:00:1772288173.567115 1812672 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
OE [2/2]: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 31668/31668 [00:02<00:00, 12035.91it/s]


========== Final Results ========================
{'step_delta': 8459,
 'test_hitrate@10': 0.12915246188640594,
 'test_hitrate@2': 0.03432486951351166,
 'test_hitrate@50': 0.3290703594684601,
 'test_loss': 0.4648432731628418,
 'test_map@10': 0.04269418492913246,
 'test_map@2': 0.02609890140593052,
 'test_map@50': 0.044256266206502914,
 'test_mrr@10': 0.0441533662378788,
 'test_mrr@2': 0.02609890140593052,
 'test_mrr@50': 0.05302617326378822,
 'test_ndcg@10': 0.06351380795240402,
 'test_ndcg@2': 0.028253011405467987,
 'test_ndcg@50': 0.10587503761053085,
 'test_precision@10': 0.015438280068337917,
 'test_precision@2': 0.017778199166059494,
 'test_precision@50': 0.010486295446753502,
 'test_recall@10': 0.016668232157826424,
 'test_recall@2': 0.0036857598461210728,
 'test_recall@50': 0.05552727356553078,
 'train_hitrate@10': 0.4837375283241272,
 'train_hitrate@2': 0.2055702954530716,
 'train_hitrate@50': 0.7927244901657104,
 'train_loss': 0.6563500761985779,
 'train_map@10': 0.20015121996

In [18]:
model.trainable_variables

[<Variable path=matrix_factorization_1/user_embedding_layer/embeddings, shape=(31669, 16), dtype=float32, value=[[ 3.5818908e-02  1.6262855e-02 -1.4128923e-02 ... -2.9658899e-03
    2.9463693e-04  2.6566599e-02]
  [-9.3695082e-02  4.0316820e-02 -1.3313115e-01 ...  1.9739599e-03
   -1.7142959e-01  7.1744129e-02]
  [-3.8174912e-01 -4.6672795e-02 -7.4999136e-01 ... -1.9449717e-01
    1.4976589e-01 -3.6457783e-01]
  ...
  [-1.5024129e-01  1.3356242e-01  9.9236727e-02 ... -1.4627782e-02
    6.8025589e-02  2.5819931e-03]
  [ 5.8986921e-02 -4.4508185e-02  2.1718860e-02 ...  1.1054015e-02
    1.5471818e-02  5.4247368e-02]
  [-2.2372220e-02  1.2812464e-01 -3.9249506e-02 ... -2.1617319e-03
   -9.5128886e-02 -1.2027067e-01]]>,
 <Variable path=matrix_factorization_1/item_embedding_layer/embeddings, shape=(38049, 16), dtype=float32, value=[[ 4.79155891e-02 -1.79821253e-03  2.66097896e-02 ...  1.39858015e-02
   -2.24042684e-04 -4.61663119e-02]
  [-9.73318040e-01  7.40625918e-01 -1.08782458e+00 ...  

In [19]:
model.user_embedding_layer.trainable_variables + model.item_embedding_layer.trainable_variables

[<Variable path=matrix_factorization_1/user_embedding_layer/embeddings, shape=(31669, 16), dtype=float32, value=[[ 3.5818908e-02  1.6262855e-02 -1.4128923e-02 ... -2.9658899e-03
    2.9463693e-04  2.6566599e-02]
  [-9.3695082e-02  4.0316820e-02 -1.3313115e-01 ...  1.9739599e-03
   -1.7142959e-01  7.1744129e-02]
  [-3.8174912e-01 -4.6672795e-02 -7.4999136e-01 ... -1.9449717e-01
    1.4976589e-01 -3.6457783e-01]
  ...
  [-1.5024129e-01  1.3356242e-01  9.9236727e-02 ... -1.4627782e-02
    6.8025589e-02  2.5819931e-03]
  [ 5.8986921e-02 -4.4508185e-02  2.1718860e-02 ...  1.1054015e-02
    1.5471818e-02  5.4247368e-02]
  [-2.2372220e-02  1.2812464e-01 -3.9249506e-02 ... -2.1617319e-03
   -9.5128886e-02 -1.2027067e-01]]>,
 <Variable path=matrix_factorization_1/item_embedding_layer/embeddings, shape=(38049, 16), dtype=float32, value=[[ 4.79155891e-02 -1.79821253e-03  2.66097896e-02 ...  1.39858015e-02
   -2.24042684e-04 -4.61663119e-02]
  [-9.73318040e-01  7.40625918e-01 -1.08782458e+00 ...  